# Cadenas de Markov y tasa de entropía

Las fuentes i.i.d. son una idealización: en la realidad, las fuentes tienen memoria. Una cadena de Markov es el modelo más sencillo con dependencia temporal: el símbolo siguiente solo depende del símbolo actual.

Este cuaderno estudia la **tasa de entropía** de una cadena de Markov y observa cómo la dependencia entre símbolos reduce la incertidumbre respecto a una fuente sin memoria.

**Artículos asociados:** [Entropía conjunta y condicional](../../02-teoria-informacion/07-entropia-conjunta-y-condicional.md), [Codificación de fuente](../../02-teoria-informacion/03-codificacion-de-fuente.md)

In [ ]:
import math
import random
from collections import Counter

def log2(x):
    return math.log2(x) if x > 0 else 0.0

print('Entorno listo.')

## Modelo: cadena de Markov homogénea

Una cadena de Markov de primer orden sobre un alfabeto $\mathcal{X}$ queda definida por:
- Una distribución **inicial** $\mu_0$ sobre $\mathcal{X}$.
- Una **matriz de transición** $P$ donde $P_{ij} = P(X_{t+1}=j \mid X_t=i)$.

La distribución **estacionaria** $\pi$ satisface $\pi P = \pi$.

In [ ]:
class MarkovChain:
    def __init__(self, states, transition):
        """
        states     : lista de estados (e.g. ['A','B','C'])
        transition : dict {estado: {estado: probabilidad}}
        """
        self.states = states
        self.P = transition
        self._validate()

    def _validate(self):
        for s in self.states:
            total = sum(self.P[s].values())
            assert math.isclose(total, 1.0, abs_tol=1e-9), f'Fila {s} no suma 1: {total}'

    def stationary(self, iterations=1000):
        """Distribución estacionaria por iteración de potencias."""
        pi = {s: 1.0 / len(self.states) for s in self.states}
        for _ in range(iterations):
            pi_new = {s: 0.0 for s in self.states}
            for i in self.states:
                for j in self.states:
                    pi_new[j] += pi[i] * self.P[i].get(j, 0.0)
            pi = pi_new
        return pi

    def entropy_rate(self):
        """Tasa de entropía H = sum_i pi_i * H(X_{t+1} | X_t = i)."""
        pi = self.stationary()
        h = 0.0
        for i in self.states:
            h_cond = -sum(p * log2(p) for p in self.P[i].values() if p > 0)
            h += pi[i] * h_cond
        return h

    def sample(self, n, start=None, seed=None):
        """Genera una secuencia de longitud n."""
        rng = random.Random(seed)
        if start is None:
            pi = self.stationary()
            start = rng.choices(self.states, weights=[pi[s] for s in self.states])[0]
        seq = [start]
        for _ in range(n - 1):
            current = seq[-1]
            nexts = list(self.P[current].keys())
            probs = list(self.P[current].values())
            seq.append(rng.choices(nexts, weights=probs)[0])
        return seq

print('Clase MarkovChain lista.')

## Ejemplo 1: cadena binaria con persistencia

El estado es 0 o 1. Con probabilidad $1-p$ se mantiene el símbolo actual; con probabilidad $p$ se cambia. Cuanto más pequeño sea $p$, mayor es la memoria y menor la tasa de entropía.

In [ ]:
def cadena_persistente(p):
    """Cadena binaria donde se cambia de estado con probabilidad p."""
    return MarkovChain(
        states=[0, 1],
        transition={
            0: {0: 1-p, 1: p},
            1: {0: p,   1: 1-p},
        }
    )

print('Tasa de entropía de la cadena persistente:')
print(f'{"p (prob. cambio)":>18}  {"H (bits/símbolo)":>18}  {"H_iid (bits/símbolo)":>22}')

for p in [0.01, 0.05, 0.1, 0.2, 0.3, 0.5]:
    mc = cadena_persistente(p)
    h_markov = mc.entropy_rate()
    # Comparar con fuente i.i.d. con la misma distribución marginal (uniforme aquí)
    h_iid = 1.0  # distribución estacionaria uniforme siempre que la cadena sea simétrica
    print(f'{p:>18.2f}  {h_markov:>18.4f}  {h_iid:>22.4f}')

print()
print('Observación: cuanto más pequeño es p, más persistente es la cadena')
print('y menor es su tasa de entropía. El caso p=0.5 es i.i.d.: H = 1 bit.')

## Ejemplo 2: modelo de clima simplificado

Tres estados: Sol (S), Nublado (N), Lluvia (L). La transición captura que el tiempo tiende a persistir.

In [ ]:
mc_clima = MarkovChain(
    states=['S', 'N', 'L'],
    transition={
        'S': {'S': 0.7, 'N': 0.2, 'L': 0.1},
        'N': {'S': 0.3, 'N': 0.4, 'L': 0.3},
        'L': {'S': 0.1, 'N': 0.3, 'L': 0.6},
    }
)

pi = mc_clima.stationary()
print('Distribución estacionaria del clima:')
for s in ['S', 'N', 'L']:
    print(f'  P({s}) = {pi[s]:.4f}')

# Entropía marginal (como si los días fueran independientes)
h_marginal = -sum(p * log2(p) for p in pi.values())
h_rate = mc_clima.entropy_rate()

print(f'\nEntropía marginal H(X_t):       {h_marginal:.4f} bits/día')
print(f'Tasa de entropía H(X_t|X_{{t-1}}): {h_rate:.4f} bits/día')
print(f'Reducción por memoria:           {h_marginal - h_rate:.4f} bits/día')

## Ejemplo 3: estimación empírica de la tasa de entropía

A partir de una secuencia larga, estimamos la tasa de entropía contando transiciones.

In [ ]:
def estimar_tasa_entropia(secuencia):
    """Estima H(X_{t+1} | X_t) a partir de los bigrams de la secuencia."""
    bigrams = Counter(zip(secuencia[:-1], secuencia[1:]))
    unigrams = Counter(secuencia[:-1])

    h = 0.0
    n = sum(unigrams.values())
    for i, count_i in unigrams.items():
        pi_i = count_i / n
        h_cond = 0.0
        for j in set(s for (a,b) in bigrams if a == i for s in [b]):
            p_ij = bigrams[(i, j)] / count_i
            h_cond -= p_ij * log2(p_ij)
        h += pi_i * h_cond
    return h

# Generar secuencias largas y estimar
print('Convergencia de la estimación empírica (cadena clima):')
print(f'{"Longitud":>10}  {"Estimada":>12}  {"Teórica":>12}')

h_teorica = mc_clima.entropy_rate()
for n in [100, 500, 2000, 10000, 50000]:
    seq = mc_clima.sample(n, seed=42)
    h_emp = estimar_tasa_entropia(seq)
    print(f'{n:>10}  {h_emp:>12.4f}  {h_teorica:>12.4f}')

## Implicación para la compresión

El teorema de Shannon para fuentes con memoria establece que la longitud media mínima de un código para $n$ símbolos de una fuente estacionaria ergódica tiende a $n \cdot H$ donde $H$ es la tasa de entropía (no la entropía marginal).

Si usamos un código diseñado para una fuente i.i.d. con la distribución marginal (ignorando la memoria), estamos usando $H(X_t)$ bits por símbolo cuando el mínimo necesario es $H(X_{t+1}|X_t)$ bits.

In [ ]:
def longitud_huffman_optima(distribucion):
    """Longitud media mínima del código de Huffman para una distribución discreta."""
    # Para una fuente discreta, la longitud media óptima está entre H y H+1.
    # Aquí calculamos el límite inferior H(X).
    return -sum(p * log2(p) for p in distribucion.values() if p > 0)

# Para la cadena del clima:
pi = mc_clima.stationary()
h_marginal = longitud_huffman_optima(pi)
h_rate = mc_clima.entropy_rate()

print('Comparación de estrategias de compresión (cadena clima):')
print(f'  Código para distribución marginal (ignora memoria): ≥ {h_marginal:.4f} bits/símbolo')
print(f'  Código que explota la memoria (Markov):             ≥ {h_rate:.4f} bits/símbolo')
print(f'  Ganancia por usar la memoria:                          {h_marginal - h_rate:.4f} bits/símbolo')
print()
print('Para 10 000 símbolos, la diferencia es:')
print(f'  {(h_marginal - h_rate) * 10000:.0f} bits extra si se ignora la memoria')

## Ideas clave

- La **tasa de entropía** de una cadena de Markov es $H = \sum_i \pi_i H(X_{t+1} \mid X_t = i)$, donde $\pi$ es la distribución estacionaria.
- La tasa de entropía es siempre $\leq$ la entropía marginal: la memoria reduce la incertidumbre.
- El caso $p = 0.5$ en la cadena binaria simétrica es i.i.d.: $H = H(X_t)$.
- Un compresor que ignora la memoria necesita más bits por símbolo que uno que la explota.
- La estimación empírica de la tasa de entropía a partir de bigrams converge con la longitud de la secuencia.